In [26]:
from pydantic import BaseModel, ConfigDict, ValidationError, model_validator, BeforeValidator, AfterValidator
from numpydantic import NDArray, Shape
from typing import Optional, Any, Annotated
import numpy as np
NpMixinT = np.lib.mixins.NDArrayOperatorsMixin
from pchandler.v2.base_arrays import BaseArray, Array4x4

In [2]:
array_N_T = NDArray[Shape["*"],Any]
array_N_3_T = NDArray[Shape["*, 3"],Any]
array_N_3_u8 = NDArray[Shape["*, 3"], np.uint8]
array_4_4_T = NDArray[Shape["4, 4"],Any]

In [3]:
n = 100000
coords = BaseArray(arr=np.random.rand(n,3))
rand = np.random.rand(n,4)

#### Overhead test on the model copy + validate method

In [4]:
%%timeit -n 100
BaseArray.model_validate(coords.model_copy(update={'xyz': rand}))

585 μs ± 22.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### Overhead test on the model_dump and re-initialise approach

In [5]:
%%timeit -n 100
b = BaseArray(**(coords.model_dump()| {'xyz': rand}))

575 μs ± 36.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Tests comparing the overhead on functions working with the object vs numpy array directly

In [6]:
n = 10000000
xyz = np.random.rand(n,3)
coords = BaseArray(arr=xyz.copy())

In [7]:
%%timeit -n 50
c = coords + 1

124 ms ± 764 μs per loop (mean ± std. dev. of 7 runs, 50 loops each)


In [8]:
%%timeit -n 50

b = xyz + 1

66.2 ms ± 343 μs per loop (mean ± std. dev. of 7 runs, 50 loops each)


In [9]:
c = coords + 1
isinstance(c, BaseArray)

True

In [43]:
from __future__ import annotations
from collections import OrderedDict
import copy
from typing import Any



class TransformRecord(OrderedDict):
    def __int__(self):
        super(TransformRecord, self).__init__()

    def __setitem__(self, key, value):
        if isinstance(value, (Array4x4, np.ndarray)):
            return super(TransformRecord, self).__setitem__(key, value)
        raise ValueError(f'Cannot set transform record value of type: {type(value)}')

    def rollback_record(self, index: int) -> tuple[Any, TransformRecord]:
        previous_history = type(self)()
        remaining_transforms = copy.deepcopy(self)

        for i in range(0, index):
            item = remaining_transforms.popitem(last=False)
            previous_history[item[0]] = item[1]

        chained_transform = remaining_transforms.popitem()[1]
        for t in remaining_transforms.items():
            chained_transform @= t[1]

        return chained_transform, previous_history

from scipy.spatial.transform import Rotation

translate = lambda x: np.array([[1, 0, 0, x[0]],
                                [0, 1, 0, x[1]],
                                [0, 0, 1, x[2]],
                                [0, 0, 0, 1]])

forward = translate(np.ones(3)*2)
backward = translate(np.ones(3)*-4)

# --------------------------------------------------
# In case a project transformation is passed
a = TransformRecord()
a['PRJ_0'] = INPUT_TRANSFORMATION
a['SOC_1'] = np.eye(4)  # then check if global shift needed ... "shouldn't" be the case...
a['REG_2'] = CustomTransformFromRegistration

# --------------------------------------------------
# No Transformation Passed
a = TransformRecord()
a['SOC_0'] = np.eye(4)
a['OPT_1'] = np.eye(4)[:3, :] - global_shift    # Where global shift needed
a['REG_2'] = CustomTransformFromRegistration    # Global shift checked
a['OPT_3'] = np.eye(4)[:3, :] - new_global_shift
a['TRANSLATED_4'] = np.eye(4)[:3, :] - arbitrary_translation

transform, old_transforms = a.rollback_record(2)

NameError: name 'INPUT_TRANSFORMATION' is not defined

In [39]:
print(transform)

[[ 1.  0.  0. -4.]
 [ 0.  1.  0. -4.]
 [ 0.  0.  1. -4.]
 [ 0.  0.  0.  1.]]


In [41]:
print(old_transforms)

TransformRecord({'Origin': array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), 'Shift_forward': array([[1., 0., 0., 2.],
       [0., 1., 0., 2.],
       [0., 0., 1., 2.],
       [0., 0., 0., 1.]])})


NameError: name 'np' is not defined